In [1]:
import numpy as np
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents

embedder = Embedder()

2026-06-28 10:21:36.158189227 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(f"Q1: First value v[0] = {v[0]:.2f}")

Q1: First value v[0] = -0.02


In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents.")

Loaded 72 documents.


In [4]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc_q2 = next(d for d in documents if d['filename'] == target_filename)

v_doc = embedder.encode(doc_q2['content'])
similarity = np.dot(v, v_doc)

print(f"Q2: Cosine similarity = {similarity:.2f}")

Q2: Cosine similarity = 0.36


In [5]:
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Created {len(chunks)} chunks.")

contents = [chunk['content'] for chunk in chunks]


X = embedder.encode_batch(contents)
X = np.array(X)


scores = X.dot(v)

best_chunk_idx = np.argmax(scores)
best_filename = chunks[best_chunk_idx]['filename']

print(f"Q3: File with highest-scoring chunk = {best_filename}")

Created 295 chunks.
Q3: File with highest-scoring chunk = 02-vector-search/lessons/07-sqlitesearch-vector.md
